# 03 -- Transfer learning: ResNet-18 and EfficientNet-B0

Both models here were pretrained on ImageNet (1.28M natural photos, 1000
object classes) and are being repurposed for spectrograms of underwater
bioacoustic calls. That's a large domain gap -- a photo of a dog and a
log-mel spectrogram of a humpback whale moan share none of the same
visual semantics -- yet transfer learning from ImageNet is a very common
and often surprisingly effective trick for spectrogram classification
tasks with limited data (which is exactly our situation for most of these
54 species: severe long-tail imbalance means many species have only a
few hundred, or even single-digit, training clips).

What actually transfers isn't "knowing what a dog looks like" -- it's
the *low-level* visual vocabulary early conv layers learn (edges,
oriented gradients, textures, blobs) which turns out to be broadly
useful for any 2D signal with local structure, spectrograms included.

Both backbones are adapted the same way (see
`watkins/models/_torchvision_backbone.py`): the single spectrogram
channel is repeated 3x to match the pretrained input, and the final
classification layer is replaced with a fresh linear layer sized for our
54 species.

In [ ]:
import os
import subprocess
import sys
import importlib.util

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Mount Drive and locate the project. Upload src/, configs/,
    #    pyproject.toml, requirements.txt to this path in My Drive first --
    #    NOT the multi-GB Watkins/ or results/ folders, those are handled
    #    separately below.
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DRIVE_PATH = "/content/drive/MyDrive/MarineMammals"  # <-- edit if you used a different path
    if not os.path.exists(f"{PROJECT_DRIVE_PATH}/src/watkins"):
        raise FileNotFoundError(
            f"Expected the project's src/ folder at {PROJECT_DRIVE_PATH}/src on Google Drive.\n"
            "Upload src/, configs/, pyproject.toml, and requirements.txt there "
            "(skip the multi-GB Watkins/ and results/ folders), or edit "
            "PROJECT_DRIVE_PATH above to match where you put them."
        )
    sys.path.insert(0, f"{PROJECT_DRIVE_PATH}/src")

    # 2. Install whatever Colab's base image doesn't already have. Deliberately
    #    does NOT touch torch/torchaudio/torchvision -- Colab's preinstalled
    #    versions are already matched to its GPU + CUDA build, and reinstalling
    #    this project's CPU-only wheels here would silently disable the GPU.
    needed = ["transformers", "timm", "soundfile", "datasets", "huggingface_hub", "pyarrow"]
    missing = [pkg for pkg in needed if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

    # 3. Data goes on fast local/ephemeral disk (re-derivable from the public
    #    Hugging Face source, no reason to pay Drive's slow random-I/O tax on
    #    thousands of per-epoch file reads); results go on Drive so trained
    #    checkpoints/metrics survive a runtime disconnect.
    os.environ["WATKINS_DATA_ROOT"] = "/content/watkins_data"
    os.environ["WATKINS_RESULTS_ROOT"] = f"{PROJECT_DRIVE_PATH}/results"

    from watkins.data import DATA_ROOT
    if not (DATA_ROOT / "manifest.csv").exists():
        print("Materializing the Watkins dataset locally -- one-time per Colab runtime, ~10-15 min...")
        subprocess.run([sys.executable, "-m", "watkins.prepare_data"], check=True)
else:
    sys.path.insert(0, "../src")

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from watkins.train import train_run, load_config
from watkins.evaluate import evaluate_checkpoint
from watkins.utils import count_parameters
from watkins.models import build_model

# Colab runs every notebook with cwd=/content regardless of where the
# .ipynb itself lives, so bare "../configs/..." paths silently miss.
# Resolve against the project root instead, which differs by environment.
PROJECT_ROOT = Path(PROJECT_DRIVE_PATH) if IN_COLAB else Path("..")


## 1. Parameter budgets, at a glance

Before training anything, look at what you're actually comparing:

In [ ]:
for name in ["baseline_cnn", "resnet18", "efficientnet_b0"]:
    kwargs = {"pretrained": True} if name != "baseline_cnn" else {}
    model, _ = build_model(name, num_classes=54, **kwargs)
    print(f"{name:<18} total={count_parameters(model):>10,}  trainable={count_parameters(model, True):>10,}")


ResNet-18 (~11.2M params) and EfficientNet-B0 (~4.1M params) both dwarf
the from-scratch baseline (~403K params) -- but only a small fraction of
that capacity is *new*; the rest arrives pretrained.

## 2. Fine-tune both (demo run)

Full configs (`configs/resnet18.yaml`, `configs/efficientnet_b0.yaml`)
fine-tune the entire backbone (`freeze_backbone: false`) with a lower
learning rate than the baseline CNN -- pretrained weights are already in
a good place, so we nudge them gently rather than the aggressive updates
a from-scratch model needs. As in notebook 02, run a fast, reduced demo
here and the full config from a terminal for the number you actually
trust.

In [ ]:
results = {}
for name, cfg_path in [("resnet18", PROJECT_ROOT / "configs/resnet18.yaml"),
                        ("efficientnet_b0", PROJECT_ROOT / "configs/efficientnet_b0.yaml")]:
    cfg = load_config(cfg_path)
    cfg["run_name"] = f"{name}_demo"
    cfg["epochs"] = 5
    cfg["subset_frac"] = 0.3
    results[name] = train_run(cfg)
    print(f"{name}: test_acc={results[name]['test_acc']:.3f}  test_f1={results[name]['test_f1']:.3f}")


## 3. Frozen backbone vs. fine-tuned: does letting ImageNet features move help or hurt?

`freeze_backbone=True` turns either model into a linear probe on top of
frozen ImageNet features -- much cheaper to train, and a useful control
condition. If frozen features do nearly as well as fine-tuning, that's
evidence the task is mostly solvable with generic visual features; if
fine-tuning clearly wins, the model is learning something spectrogram-
specific that ImageNet never taught it.

In [ ]:
cfg_frozen = load_config(PROJECT_ROOT / "configs/resnet18.yaml")
cfg_frozen["run_name"] = "resnet18_frozen_demo"
cfg_frozen["epochs"] = 5
cfg_frozen["subset_frac"] = 0.3
cfg_frozen["model_kwargs"] = {"pretrained": True, "freeze_backbone": True}
cfg_frozen["lr"] = 1e-2  # frozen backbone -> only the head trains, can afford a higher LR

frozen_result = train_run(cfg_frozen)
print(f"resnet18 (frozen backbone): test_acc={frozen_result['test_acc']:.3f}  test_f1={frozen_result['test_f1']:.3f}")
print(f"resnet18 (fine-tuned):      test_acc={results['resnet18']['test_acc']:.3f}  test_f1={results['resnet18']['test_f1']:.3f}")


## 4. Evaluate and compare confusion matrices

In [ ]:
for name in ["resnet18", "efficientnet_b0"]:
    evaluate_checkpoint(results[name]["checkpoint_path"])


## Exercises

1. Run the frozen-vs-fine-tuned comparison for EfficientNet-B0 too. Is
   the frozen/fine-tuned gap similar in size to ResNet-18's, or does it
   differ? What might that tell you about how each architecture's
   pretrained features generalize?
2. `_torchvision_backbone.py` repeats the single spectrogram channel 3x
   rather than resizing the first conv layer to accept 1 channel
   natively. Try implementing the alternative (average the pretrained
   first-conv weights across the 3 input channels to initialize a
   1-channel conv) and compare -- does it change results meaningfully?
3. Compare training wall-clock time (`seconds` column in the training
   log CSV) between the baseline CNN, frozen ResNet-18, and fine-tuned
   ResNet-18. Is the accuracy gained from fine-tuning worth the extra
   compute, on this dataset size?
4. Run the full (non-demo) configs to completion and update the
   comparison table you'll build in notebook 05.